# Itinerary recommendation — run on Colab (self-contained)

Runs **Strategy A** (frozen next-POI rollout) and trains **Strategy B** (pointer network), then compares them.

**Prerequisite:** Phase-1 already produced these on your Drive (from `train_poi.ipynb`):
- `/MyDrive/poi-rec/checkpoints/NYC/best.pt`
- `/MyDrive/poi-rec/data/processed/NYC/` (train/val/test.parquet, edge_index.pt, poi_coords.npy, meta.json)

**How to run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**. Approve the Google Drive auth popup when cell 2 asks. That's the only manual step.

## 1. Install dependencies

In [6]:
!pip install -q torch_geometric pyarrow
print('deps installed')

deps installed


## 2. Mount Drive  (approve the popup)

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/poi-rec'
assert os.path.isdir(PROJECT_ROOT), 'poi-rec not found on Drive — run Phase 1 first'
print('Drive mounted; PROJECT_ROOT =', PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted; PROJECT_ROOT = /content/drive/MyDrive/poi-rec


## 3. Get the code + set device/seed

In [8]:
import sys, subprocess, importlib, random
REPO_URL = 'https://github.com/6ym6n/PFE_IMPLEMTATION.git'
REPO_DIR = '/content/PFE_IMPLEMTATION'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--quiet',REPO_URL,REPO_DIR], check=True)
else:
    subprocess.run(['git','-C',REPO_DIR,'pull','--quiet'], check=False)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
for m in [k for k in list(sys.modules) if k=='src' or k.startswith('src.')]: del sys.modules[m]
importlib.invalidate_caches()

import numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo ready | DEVICE =', DEVICE)

repo ready | DEVICE = cuda


## 4. Sanity check (artifacts present + GPU)

In [9]:
import torch
print('GPU      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (set Runtime->T4)')
print('best.pt  :', os.path.isfile(f'{PROJECT_ROOT}/checkpoints/NYC/best.pt'))
print('processed:', os.path.isdir(f'{PROJECT_ROOT}/data/processed/NYC'))

GPU      : Tesla T4
best.pt  : True
processed: True


## 5. Strategy A — frozen next-POI rollout (no training, ~minutes)
Decodes greedy + beam itineraries from the frozen `best.pt`. Writes `results/NYC_itinerary_{greedy,beam}.json`.

In [10]:
from src.itinerary.run_itinerary import run_itinerary
itin_A = run_itinerary('NYC', project_root=PROJECT_ROOT, device=DEVICE, beam=3)
print('Strategy A len>=3 pairs-F1: greedy=%.4f | beam3=%.4f'
      % (itin_A['greedy']['len_ge_3']['pairs-F1'], itin_A['beam']['len_ge_3']['pairs-F1']))

[NYC] 5726 itinerary queries (length>=3: 2880) | assumed_dt_hours=2.484
[NYC] greedy | ALL      : pairs-F1=0.5808 | set-F1=0.7828 | exact-match=0.4628 | feasibility=1.0000 | n=5726
[NYC] greedy | len>=3   : pairs-F1=0.2887 | set-F1=0.6089 | exact-match=0.0542 | feasibility=1.0000 | n=2880
[NYC] beam   | ALL      : pairs-F1=0.5815 | set-F1=0.7834 | exact-match=0.4640 | feasibility=1.0000 | n=5726
[NYC] beam   | len>=3   : pairs-F1=0.2902 | set-F1=0.6101 | exact-match=0.0566 | feasibility=1.0000 | n=2880
Strategy A len>=3 pairs-F1: greedy=0.2887 | beam3=0.2902


## 6. Strategy B — train the pointer network (~30-60 min on T4)
Trains a NEW model on whole length≥3 trajectories, early-stops on val pairs-F1. Does NOT touch `best.pt`. Writes `checkpoints/NYC_pointer/` and `results/NYC_pointer_test.json`.

> If the runtime disconnects mid-training, just re-run this cell — it resumes from a fresh start but the data/checkpoints persist on Drive.

In [11]:
from src.itinerary.train_pointer import train_pointer_model
model_B, test_B, history_B = train_pointer_model(
    'NYC', project_root=PROJECT_ROOT, device=DEVICE,
    epochs=50, patience=8, beam=3, min_len=3,
)


Strategy-B pointer training on NYC
pointer train sessions=10,281 | val queries=1,093 | test queries=2,880
Parameters: 924,480


train-ptr:   0%|          | 0/161 [00:00<?, ?it/s]

[NYC] epoch 01 | loss=8.0114 | val: pairs-F1=0.1644 | set-F1=0.4704 | exact-match=0.0027 | feasibility=1.0000 | n=1093


train-ptr:   0%|          | 0/161 [00:00<?, ?it/s]

[NYC] epoch 02 | loss=7.2976 | val: pairs-F1=0.1701 | set-F1=0.4806 | exact-match=0.0018 | feasibility=1.0000 | n=1093


...

[NYC] epoch 34 | loss=3.4633 | val: pairs-F1=0.2664 | set-F1=0.5849 | exact-match=0.0448 | feasibility=1.0000 | n=1093


train-ptr:   0%|          | 0/161 [00:00<?, ?it/s]

[NYC] epoch 35 | loss=3.4355 | val: pairs-F1=0.2696 | set-F1=0.5877 | exact-match=0.0485 | feasibility=1.0000 | n=1093


train-ptr:   0%|          | 0/161 [00:00<?, ?it/s]

[NYC] epoch 36 | loss=3.4147 | val: pairs-F1=0.2697 | set-F1=0.5888 | exact-match=0.0448 | feasibility=1.0000 | n=1093
Early stop at epoch 36. Best epoch 28 (val pairs-F1=0.2713)

[NYC] TEST greedy (best ep 28): pairs-F1=0.2585 | set-F1=0.5783 | exact-match=0.0434 | feasibility=1.0000 | n=2880
[NYC] TEST beam3: pairs-F1=0.2585 | set-F1=0.5787 | exact-match=0.0434 | feasibility=1.0000 | n=2880


## 7. Compare A vs B  (length≥3 pairs-F1 is the headline)

In [12]:
import json
def _load(p):
    fp = f'{PROJECT_ROOT}/results/{p}'
    return json.load(open(fp)) if os.path.exists(fp) else None
A = _load('NYC_itinerary_greedy.json'); B = _load('NYC_pointer_test.json')
print('=== NYC itinerary — length>=3 pairs-F1 ===')
if A: print('  Strategy A (frozen rollout, greedy): %.4f' % A['len_ge_3']['pairs-F1'])
if B:
    bk = [k for k in B if k.startswith('beam')][0]
    print('  Strategy B (pointer, greedy)        : %.4f' % B['greedy']['pairs-F1'])
    print('  Strategy B (pointer, %s)          : %.4f' % (bk, B[bk]['pairs-F1']))
    if A:
        print('  --> B beats the floor by: %+.4f pairs-F1' % (B['greedy']['pairs-F1'] - A['len_ge_3']['pairs-F1']))

=== NYC itinerary — length>=3 pairs-F1 ===
  Strategy A (frozen rollout, greedy): 0.2887
  Strategy B (pointer, greedy)        : 0.2585
  Strategy B (pointer, beam3)          : 0.2585
  --> B beats the floor by: -0.0302 pairs-F1
